In [7]:
import pandas as pd

df = pd.read_csv("../data/processed/cleaned_online_retail.csv")
pairs_df = pd.read_csv("../data/processed/product_pairs.csv")

C:\Users\HP\AppData\Local\Temp\ipykernel_700\3385076292.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/cleaned_online_retail.csv")


In [8]:
order_frequency = (
    df.groupby("StockCode")["InvoiceNo"]
      .nunique()
)

In [9]:
total_quantity = (
    df.groupby("StockCode")["Quantity"]
      .sum()
)

In [10]:
avg_quantity = (
    df.groupby(["StockCode", "InvoiceNo"])["Quantity"]
      .sum()
      .groupby("StockCode")
      .mean()
)

In [11]:
product_features = pd.DataFrame({
    "order_frequency": order_frequency,
    "total_quantity": total_quantity,
    "avg_quantity_per_order": avg_quantity
})

In [12]:
product_features.head()

,order_frequency,total_quantity,avg_quantity_per_order
StockCode,,,
10002,71,860,12.112676
10080,22,303,13.772727
10120,29,192,6.620690
10123C,3,5,1.666667
10124A,5,16,3.200000


In [13]:
pairs_df.head()

,Product_Pair,Order_Count,support
0,"('22386', '85099B')",825,0.041329
1,"('22697', '22699')",767,0.038423
2,"('21931', '85099B')",724,0.036269
3,"('22411', '85099B')",680,0.034065
4,"('20725', '22383')",655,0.032812


In [15]:
import ast

pairs_df["Product_Pair"] = pairs_df["Product_Pair"].apply(ast.literal_eval)



In [16]:
related_products = {}

for pair in pairs_df["Product_Pair"]:
    product_a, product_b = pair
    
    related_products.setdefault(product_a, set()).add(product_b)
    related_products.setdefault(product_b, set()).add(product_a)

In [17]:
product_features["related_product_count"] = (
    product_features.index
    .map(lambda x: len(related_products.get(x, set())))
)

In [18]:
product_features.head(10)

,order_frequency,total_quantity,avg_quantity_per_order,related_product_count
StockCode,,,,
10002,71,860,12.112676,2067
10080,22,303,13.772727,908
10120,29,192,6.620690,1001
10123C,3,5,1.666667,85
10124A,5,16,3.200000,196
10124G,4,17,4.250000,199
10125,91,1295,14.230769,2373
10133,196,2856,14.571429,2732
10135,175,2229,12.737143,2893


In [19]:
product_features.isnull().sum()

order_frequency           0
total_quantity            0
avg_quantity_per_order    0
related_product_count     0
dtype: int64

In [20]:
product_features.describe()

,order_frequency,total_quantity,avg_quantity_per_order,related_product_count
count,3922.000000,3922.000000,3922.000000,3922.000000
mean,132.484192,1420.810811,29.417691,1912.547680
std,193.890513,3579.835163,1293.401511,1006.136593
min,1.000000,1.000000,1.000000,0.000000
25%,16.000000,54.000000,2.600000,1103.250000
50%,65.000000,369.500000,5.413199,2162.000000
75%,165.000000,1393.750000,10.229681,2759.750000
max,2198.000000,80995.000000,80995.000000,3562.000000


In [21]:
product_features.to_csv(
    "../data/processed/product_features.csv"
)